# 05 — 좌우 분할 피처 해상도 측정 + Trackman 초과분 정체

`EDA_정리.md`의 "다음 단계 #1"과 "미해결 #3, #5"를 처리한다. 모델 학습 없음
(노트북 04와 동일 방법론: strictly-prior 누적 피처 → 분위 20구간 + EB 축소(k=200) →
out-of-time 평가). Colab 이 아닌 **로컬 환경**에서 실행 (`DATA_DIR`, `ART_DIR` 경로만 다름).

1. `asof_success_R_same` / `_diff` / `_platoon` / `_platoon_blend`: 투수 개인 이력을
   "타자 손잡이가 투수와 같은가"로 나눈 strictly-prior 누적 성공률이 `asof_success_R2`(517.5)를
   넘는지 4개 폴드(A/B/C/D)에서 확인한다.
2. `trackman_history.csv` 행 수가 main R(1,314,088)보다 많은 이유(미해결 #5)를 확인한다.

In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.width", 190)
pd.set_option("display.max_columns", 80)

DATA_DIR = "../data"
TARGET = "control_success"


## 1. 좌우 분할 피처

### 1-1. 피처 구성

`add_prior_rate`는 노트북 04 §3에서 `asof_success_R2`를 만든 것과 **동일한 함수**다
(strictly-prior: `cumsum() - 자기값`). 마스크만 다르게 줘서

- `asof_success_R_same`: R 전용 + **타자 손잡이가 투수와 같을 때만** 누적
- `asof_success_R_diff`: R 전용 + **다를 때만** 누적
- `asof_success_R_platoon`: 현재 행의 매치업에 맞는 쪽을 선택 (same → `_same`, diff → `_diff`)
- `asof_success_R_platoon_blend`: `asof_success_R_platoon`을 표본 수 기반으로
  `asof_success_R2` 쪽으로 축소(credibility `k=30`) — 콜드 셀 보정

In [2]:
cols = ["row_id", "season", "game_type", "pitcher_id", "batter_id", "pitcher_hand",
        "batter_hand", "asof_pitcher_n", "asof_pitcher_success_rate", TARGET]
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), usecols=cols)

P = train.sort_values(["pitcher_id", "asof_pitcher_n"], kind="mergesort").reset_index(drop=True)


def add_prior_rate(df, src, name, mask=None):
    v = df[src].to_numpy(float)
    ok = np.isfinite(v)
    m = np.ones(len(df)) if mask is None else mask.astype(float)
    use = ok.astype(float) * m
    val = np.where(ok, v, 0.0) * m
    tmp = pd.DataFrame({"g": df["pitcher_id"].values, "u": use, "v": val})
    cu = tmp.groupby("g")["u"].cumsum() - tmp["u"]
    cv = tmp.groupby("g")["v"].cumsum() - tmp["v"]
    df[name] = np.where(cu > 0, cv / cu, np.nan)
    df[name + "_n"] = cu
    return df


maskR = (P.game_type == "R").to_numpy()
same_hand = (P.pitcher_hand == P.batter_hand).to_numpy()

P = add_prior_rate(P, TARGET, "asof_success_R2", mask=maskR)
P = add_prior_rate(P, TARGET, "asof_success_R_same", mask=maskR & same_hand)
P = add_prior_rate(P, TARGET, "asof_success_R_diff", mask=maskR & (~same_hand))

P["asof_success_R_platoon"] = np.where(same_hand, P["asof_success_R_same"], P["asof_success_R_diff"])
P["asof_success_R_platoon_n"] = np.where(same_hand, P["asof_success_R_same_n"], P["asof_success_R_diff_n"])

K_BLEND = 30.0
n_platoon = P["asof_success_R_platoon_n"].to_numpy(float)
plat = P["asof_success_R_platoon"].to_numpy(float)
base = P["asof_success_R2"].to_numpy(float)
plat_filled = np.where(np.isfinite(plat), plat, base)
w = np.where(np.isfinite(n_platoon), n_platoon / (n_platoon + K_BLEND), 0.0)
P["asof_success_R_platoon_blend"] = np.where(np.isfinite(base), w * plat_filled + (1 - w) * base, plat_filled)

print("신규 피처:", [c for c in P.columns if c.startswith("asof_success_R")])

신규 피처: ['asof_success_R2', 'asof_success_R2_n', 'asof_success_R_same', 'asof_success_R_same_n', 'asof_success_R_diff', 'asof_success_R_diff_n', 'asof_success_R_platoon', 'asof_success_R_platoon_n', 'asof_success_R_platoon_blend']


### 1-2. 4개 폴드에서 해상도 측정

노트북 03/04와 동일한 `_score_pair` / `feature_resolution`을 그대로 가져온다
(오라클 절편 보정 후 = 해상도만, out-of-time이라 낙관 편향 없음).

In [3]:
def _score_pair(p, yv):
    p = np.asarray(p, float); yv = np.asarray(yv, float)
    r = yv.mean(); den = r * (1 - r)
    bias = p.mean() - r
    pts_raw = 1e5 * (1 - np.mean((p - yv) ** 2) / den)
    pts_res = 1e5 * (1 - np.mean((p - bias - yv) ** 2) / den)
    return float(pts_res), float(pts_raw), float(bias)


def feature_resolution(tr, va, col, n_bins=20, k_shrink=200.0):
    xt = tr[col].to_numpy(float); xv = va[col].to_numpy(float)
    fin = np.isfinite(xt)
    if fin.sum() < 1000:
        return None
    edges = np.unique(np.nanquantile(xt[fin], np.linspace(0, 1, n_bins + 1)))
    if len(edges) < 3:
        return None
    def assign(v):
        b = np.digitize(v, edges[1:-1]).astype(float); b[~np.isfinite(v)] = -1.0; return b
    bt, bv = assign(xt), assign(xv)
    yt = tr[TARGET].to_numpy(float); yv = va[TARGET].to_numpy(float)
    prior = yt.mean()
    g = pd.DataFrame({"b": bt, "y": yt}).groupby("b")["y"].agg(["sum", "size"])
    lut = ((g["sum"] + prior * k_shrink) / (g["size"] + k_shrink)).to_dict()
    p = np.array([lut.get(b, prior) for b in bv])
    pts_res, pts_raw, bias = _score_pair(p, yv)
    return dict(col=col, pts_res=pts_res, pts_raw=pts_raw, bias=bias,
                n_bins=len(edges) - 1, nan_pct=round(float((~np.isfinite(xv)).mean() * 100), 2))


FOLDS = {
    "A(2019-21->2022)": (2019, 2021, 2022),
    "B(2020-22->2023)": (2020, 2022, 2023),
    "C(2021-23->2024)": (2021, 2023, 2024),
    "D(2019-23->2024)": (2019, 2023, 2024),
}
CANDS = ["asof_pitcher_success_rate", "asof_success_R2",
         "asof_success_R_same", "asof_success_R_diff",
         "asof_success_R_platoon", "asof_success_R_platoon_blend"]

all_res = {}
for label, (tr_lo, tr_hi, va_yr) in FOLDS.items():
    tr = P[P.season.between(tr_lo, tr_hi) & (P.game_type == "R")]
    va = P[(P.season == va_yr) & (P.game_type == "R")]
    rows = [feature_resolution(tr, va, c) for c in CANDS]
    res = pd.DataFrame([r for r in rows if r]).set_index("col")
    all_res[label] = res["pts_res"]
    print(f"=== {label}  tr={len(tr):,}  va={len(va):,} ===")
    print(res.to_string())
    print()

summary = pd.DataFrame(all_res)
summary["mean"] = summary.mean(axis=1)
print("=== pts_res 폴드별 요약 (오라클 절편 보정, 해상도만) ===")
print(summary.round(1).to_string())

=== A(2019-21->2022)  tr=653,728  va=217,024 ===
                                 pts_res     pts_raw      bias  n_bins  nan_pct
col                                                                            
asof_pitcher_success_rate     363.543241  304.265486  0.012173      20     0.02
asof_success_R2               505.638900  469.737227  0.009474      20     0.03
asof_success_R_same           414.479928  354.474116  0.012248      20     0.11
asof_success_R_diff           347.085474  272.005325  0.013700      20     0.18
asof_success_R_platoon        576.111388  529.866814  0.010752      20     0.06
asof_success_R_platoon_blend  590.900006  549.342588  0.010193      20     0.03



=== B(2020-22->2023)  tr=659,125  va=219,839 ===
                                 pts_res     pts_raw      bias  n_bins  nan_pct
col                                                                            
asof_pitcher_success_rate     428.898096  428.051512  0.001455      20     0.01
asof_success_R2               489.691184  489.353817 -0.000918      20     0.02
asof_success_R_same           421.481664  420.669080  0.001425      20     0.10
asof_success_R_diff           385.907836  383.276087  0.002565      20     0.11
asof_success_R_platoon        570.251740  570.166541  0.000462      20     0.05
asof_success_R_platoon_blend  581.864353  581.842753  0.000232      20     0.02



=== C(2021-23->2024)  tr=658,090  va=223,497 ===
                                 pts_res     pts_raw      bias  n_bins  nan_pct
col                                                                            
asof_pitcher_success_rate     434.571139  420.471188  0.005936      20     0.02
asof_success_R2               517.450355  505.610793  0.005439      20     0.03
asof_success_R_same           378.099702  357.460845  0.007182      20     0.13
asof_success_R_diff           417.800563  387.303572  0.008730      20     0.12
asof_success_R_platoon        603.132114  587.590450  0.006232      20     0.05
asof_success_R_platoon_blend  614.100184  600.495115  0.005831      20     0.03



=== D(2019-23->2024)  tr=1,090,591  va=223,497 ===
                                 pts_res     pts_raw      bias  n_bins  nan_pct
col                                                                            
asof_pitcher_success_rate     436.722033  385.975746  0.011261      20     0.02
asof_success_R2               519.921852  487.615487  0.008985      20     0.03
asof_success_R_same           374.517183  317.825260  0.011903      20     0.13
asof_success_R_diff           430.320752  334.556018  0.015470      20     0.12
asof_success_R_platoon        602.192728  553.855357  0.010991      20     0.05
asof_success_R_platoon_blend  615.800898  573.671296  0.010261      20     0.03

=== pts_res 폴드별 요약 (오라클 절편 보정, 해상도만) ===
                              A(2019-21->2022)  B(2020-22->2023)  C(2021-23->2024)  D(2019-23->2024)   mean
col                                                                                                        
asof_pitcher_success_rate                363.5     

### 1-3. 콜드스타트 심각도

`_platoon`은 이력을 반으로 쪼개므로 표본이 부족한 셀이 걱정된다.
2024 R 검증셋에서 `asof_success_R_platoon_n`(현재 매치업 쪽 누적 표본 수) 분포를 본다.

In [4]:
va = P[(P.season == 2024) & (P.game_type == "R")]
n_platoon = va["asof_success_R_platoon_n"]
print("n rows:", len(va))
print("platoon_n 분포:")
print(n_platoon.describe().round(1))
print()
print("pct platoon_n < 10 :", round((n_platoon < 10).mean() * 100, 2))
print("pct platoon_n < 30 :", round((n_platoon < 30).mean() * 100, 2))
print("pct platoon rate NaN (표본 0) :", round(va['asof_success_R_platoon'].isna().mean() * 100, 2))

n rows: 223497
platoon_n 분포:
count    223497.0
mean       2062.8
std        1875.0
min           0.0
25%         572.0
50%        1495.0
75%        3025.0
max        8379.0
Name: asof_success_R_platoon_n, dtype: float64

pct platoon_n < 10 : 0.52
pct platoon_n < 30 : 1.54
pct platoon rate NaN (표본 0) : 0.05


### 1-4. 검증

`asof_success_R2`, `asof_pitcher_success_rate`의 fold C 값이 노트북 04의 517.5 / 434.6과
일치하는지 확인 — 방법론이 그대로 이식됐다는 근거. (위 §1-2 출력의 `C(2021-23->2024)` 블록 참조.
`asof_success_R2` 517.450, `asof_pitcher_success_rate` 434.571로 소수 첫째 자리까지 일치.)

수작업 검산: 특정 투수 1명을 뽑아 `asof_success_R_same`를 for-loop로 직접 계산해
벡터화 결과와 동일한지 확인 (strictly-prior 위반 여부 체크).

In [5]:
pid = P.pitcher_id.value_counts().index[0]
subR = P[(P.pitcher_id == pid) & (P.game_type == "R")].reset_index(drop=True)
same = (subR.pitcher_hand == subR.batter_hand)
y = subR[TARGET].to_numpy()

manual = np.full(len(subR), np.nan)
cs, cn = 0, 0
for i in range(len(subR)):
    manual[i] = cs / cn if cn > 0 else np.nan
    if same.iloc[i]:
        cs += y[i]; cn += 1

vec = subR["asof_success_R_same"].to_numpy()
mismatch = np.nansum(np.abs(manual - vec) > 1e-9)
print(f"pitcher {pid}  R rows={len(subR)}  same-hand rows={int(same.sum())}")
print("수작업 vs 벡터화 불일치 개수:", mismatch)

pitcher 23633  R rows=15352  same-hand rows=7601
수작업 vs 벡터화 불일치 개수: 0


### 1-5. 결론

| 피처 | A | B | C | D | 비고 |
|---|---|---|---|---|---|
| `asof_pitcher_success_rate` (주최) | 363.5 | 428.9 | 434.6 | 436.7 | 기준 |
| `asof_success_R2` (기존 최고, R전용) | 505.6 | 489.7 | **517.5** | 519.9 | 노트북 04 결과와 일치 |
| `asof_success_R_platoon` | 576.1 | 570.3 | 603.1 | 602.2 | R2 대비 +70~+86 |
| **`asof_success_R_platoon_blend`** | **590.9** | **581.9** | **614.1** | **615.8** | R2 대비 **+85~+97**, 전 폴드 동일 부호 |

- **4개 폴드 전부에서 `asof_success_R2`를 크게 앞선다** (부호 일관성 충족).
- 콜드스타트는 무시할 수준: 2024 검증셋에서 `platoon_n` 중앙값 1,495,
  `< 30`인 행 1.54%, 완전 결측(표본 0) 0.05%.
- `_same`/`_diff` 단독은 각각 R2보다 낮다(반쪽 이력만 쓰므로 당연) — **매치업에 맞게 선택하는
  `_platoon`으로 합쳐야** 의미가 있다.
- 셀 세분화가 항상 손해였던 카운트 분할(−30.9, 노트북 04 §4)과 달리 손잡이 분할은 크게 이득 —
  이전 그룹 lookup 실험의 "좌우 분할 +79.4"(노트북 04 §7, `pitcher × batter_hand`)와 방향은
  같지만, strictly-prior 연속 누적 방식이 고정 셀 lookup보다 자유도가 낮아 결과가 더 크다.
- **다음 단계**: 이 피처를 실제 모델(LightGBM)에 넣고 `pitcher × batter_hand` group lookup과
  중복 정보인지, 상호 보완적인지 폴드 짝지은 Δ로 확인.

## 2. Trackman 초과분의 정체 (미해결 #5)

`trackman_history.csv` = 1,793,078행인데 main `train` R(1,314,088) 또는 전체(1,475,092)보다
많다. `pitcher_team` 코드를 까 보면 답이 나온다.

In [6]:
tm = pd.read_csv(os.path.join(DATA_DIR, "trackman_history.csv"),
                  usecols=["season", "pitcher_team"])
vc = tm["pitcher_team"].value_counts()
print(vc)
print()

is_farm = vc.index.str.startswith("MIN_")
is_special = vc.index.isin(["KBO_ARM", "KBO_POL", "ACE_MEX"])
is_first = ~is_farm & ~is_special

print("1군(팀 전체) 코드 합:", vc[is_first].sum())
print("2군(MIN_*, 팀 전체) 코드 합:", vc[is_farm].sum())
print("기타(상무/경찰/교육리그 등) 코드 합:", vc[is_special].sum())
print("총합:", vc.sum())

pitcher_team
DOO_BEA    128074
NC_DIN     127091
LG_TWI     125354
HAN_EAG    124574
KIW_HER    124418
KT_WIZ     123456
LOT_GIA    123357
SAM_LIO    122442
SSG_LAN     84767
MIN_NCD     68421
MIN_DOO     65587
MIN_HAN     65442
MIN_SAM     65328
MIN_LGT     64449
KIA_TIG     63972
MIN_LOT     61384
MIN_KTW     55823
MIN_SSG     47078
MIN_HER     43898
SK_WYV      41895
MIN_KIA     29931
KBO_ARM     25776
MIN_SKW      9215
KBO_POL       902
MIN_HAW       292
ACE_MEX       152
Name: count, dtype: int64

1군(팀 전체) 코드 합: 1189400
2군(MIN_*, 팀 전체) 코드 합: 576848
기타(상무/경찰/교육리그 등) 코드 합: 26830
총합: 1793078


### 결론

`trackman_history.csv`의 `pitcher_team`은 **26개 코드**로, main 데이터의 팀 ID 13개(1군 10개 +
2군 특수 3개: 상무/경찰/팀13 2군)보다 훨씬 세분화돼 있다.

- **1군 10개 구단** 전부 (`KIA_TIG`, `KT_WIZ`, `LG_TWI`, ... ) — main의 `game_type=R`과 대응
- **2군 10개 구단 전부** (`MIN_DOO`, `MIN_HAN`, ... `MIN_*` 접두어) — main의 `game_type=F`는
  이 중 **팀13(`MIN_KTW`) 관련 경기만** 포함 (`EDA_정리.md` §4 "F는 팀13 조직 경기만").
  나머지 9개 구단의 2군 경기는 main에 전혀 없고 trackman에만 있다.
- 상무(`KBO_ARM`)·경찰(`KBO_POL`)·교육리그로 추정되는 `ACE_MEX`/`MIN_HAW` 소량

**즉 trackman은 리그 전체(1군+전 구단 2군)를 다루는 반면, main 데이터의 F는 "우리 구단(팀13)이
낀 2군 경기"로만 필터링됐다.** 초과분(1,793,078 − 1,475,092 ≈ 318K행)의 대부분은
main에 없는 **다른 9개 구단의 2군 경기**다. 대회 규칙상 trackman은 참가자가 파생 피처를
만드는 참고 자료일 뿐 main과 1:1 결합 테이블이 아니므로, 이 정체 확인은 "trackman 기반
투수별 요약 피처를 만들 가치가 있는가"를 판단하는 데는 쓰이지만 실행 우선순위는 낮게 유지한다
(ID 공간 불일치 + 2025 데이터 미제공이라는 기존 결론은 그대로).

## 3. 요약

In [7]:
print("=" * 78)
print("1) 좌우 분할 피처 asof_success_R_platoon_blend가 4개 폴드 전부에서")
print("   asof_success_R2(517.5, 기존 최고)를 +85~+97점 앞선다.")
print("   -> 다음 EDA 단계에서 최우선 피처 후보. 콜드스타트 영향 미미(NaN 0.05%).")
print()
print("2) trackman_history 초과분 = 리그 전체 2군(9개 non-팀13 구단) 커버리지.")
print("   main의 F는 팀13 관련 경기만 필터링된 부분집합. 우선순위는 낮게 유지.")

1) 좌우 분할 피처 asof_success_R_platoon_blend가 4개 폴드 전부에서
   asof_success_R2(517.5, 기존 최고)를 +85~+97점 앞선다.
   -> 다음 EDA 단계에서 최우선 피처 후보. 콜드스타트 영향 미미(NaN 0.05%).

2) trackman_history 초과분 = 리그 전체 2군(9개 non-팀13 구단) 커버리지.
   main의 F는 팀13 관련 경기만 필터링된 부분집합. 우선순위는 낮게 유지.
